# Scoring des baselines (direct llm / cot) -- 500 scenarios x 5 modeles x 3 strategies x 2 baselines

Ces deux baselines font tout en UN appel LLM (extraction + decision + placement,
pas de necessity_checker ni de solveur CP-SAT dedie) -- voir
eval/baselines/direct llm/prompt.py et eval/baselines/cot/prompt.py.

Ce que ce notebook calcule, par enregistrement :
1. **Succes de parsing** -- le modele a-t-il produit un JSON valide contre le schema attendu.
2. **Exactitude de la decision** (STAY/OFFLOAD/INFEASIBLE) vs `ground_truth.decision`.
3. **Validite REELLE du placement propose** -- verifie avec `check_candidate_placement`
   (la meme fonction que necessity_checker/decision_engine utilisent en production)
   contre les VRAIES exigences du scenario, pas celles que le modele a extraites.
4. **Exactitude de l'ensemble des services deplaces**, seulement quand la verite
   terrain dit OFFLOAD.

Le placement propose par le modele est souvent incomplet (seuls les services
qui bougent sont parfois lists) -- on le complete avec le placement courant
pour les services omis, exactement comme le faisait l'ancien run_sample.py
avant la simplification "capture raw, score later".


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parents[1]  # repo root, assuming this notebook runs from eval/baselines/
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval" / "system_model"))

from agent.model.schemas import Requirement  # noqa: E402
from agent.utils.pipeline import check_candidate_placement  # noqa: E402
from loader import load_flows, load_links, load_nodes, load_services  # noqa: E402

RESULTS_DIR = ROOT / "eval" / "baselines" / "results"
KPI_REMAP = {"e2e_latency": "latency"}

MODEL_SHORT_NAME = {
    "google/gemini-2.5-flash-lite": "gemini-2.5-flash-lite",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral-small-3.2-24b",
    "openai/gpt-oss-20b": "gpt-oss-20b",
    "meta-llama/llama-3.3-70b-instruct": "llama-3.3-70b",
    "openai/gpt-oss-120b": "gpt-oss-120b",
}
MODEL_ORDER = ["gemini-2.5-flash-lite", "gpt-oss-120b", "gpt-oss-20b", "llama-3.3-70b", "mistral-small-3.2-24b"]
STRATEGY_ORDER = ["zero_shot", "one_shot", "few_shot"]
STRATEGY_COLORS = {"zero_shot": "#3569A8", "one_shot": "#28A69A", "few_shot": "#E99A43"}
STRATEGY_LABELS = {"zero_shot": "Zero-shot", "one_shot": "One-shot", "few_shot": "Few-shot"}


## Chargement -- un enregistrement par (scenario, modele, strategie, baseline)

In [ ]:
with open(ROOT / "eval" / "experiment_500" / "scenarios.json") as f:
    SCENARIOS_BY_ID = {s["id"]: s for s in json.load(f)}

NODES = load_nodes()
LINKS = load_links()
_fixtures_cache = {}


def scenario_fixtures(scenario_id):
    if scenario_id not in _fixtures_cache:
        scenario = SCENARIOS_BY_ID[scenario_id]
        services = load_services(scenario)
        flows = load_flows(scenario)
        gt_reqs = []
        for i, req in enumerate(scenario["requirements"]):
            kpi = KPI_REMAP.get(req["kpi_type"], req["kpi_type"])
            gt_reqs.append(Requirement(
                kpi_type=kpi, comparator=req["comparator"], target_value=req["new_value"],
                unit=req["unit"], target_type=req["target_type"], target_id=req["target_id"],
                source_span="", requirement_id=f"gt-{i}",
            ))
        baseline_placement = {sid: s.current_node for sid, s in services.items()}
        _fixtures_cache[scenario_id] = (services, flows, gt_reqs, baseline_placement)
    return _fixtures_cache[scenario_id]


def full_placement(predicted_placement, baseline_placement):
    """Backfill any service the model omitted with its current_node -- an
    incomplete placement is a formatting slip, not grounds to treat the
    whole answer as unusable."""
    if predicted_placement is None:
        return dict(baseline_placement)
    return {sid: predicted_placement.get(sid, node) for sid, node in baseline_placement.items()}


records = []
for f in sorted(RESULTS_DIR.glob("*.jsonl")):
    with open(f) as fh:
        for line in fh:
            r = json.loads(line)
            if "error" not in r:
                records.append(r)

print(f"{len(records)} records loaded (of 15000 attempted)")


## Construction du tableau de scoring

In [ ]:
rows = []
for r in records:
    rc = r["run_config"]
    result = r.get("result")
    services, flows, gt_reqs, baseline_placement = scenario_fixtures(r["scenario_id"])

    parsed_ok = result is not None
    pred_decision = result.get("decision") if result else None
    pred_placement_raw = result.get("placement") if result else None
    pred_full_placement = full_placement(pred_placement_raw, baseline_placement)
    pred_moved = frozenset(sid for sid, node in pred_full_placement.items() if node != baseline_placement[sid])
    gt_moved = frozenset(f"C{n}" for n in r["ground_truth"]["moved_components"])
    gt_decision = r["ground_truth"]["decision"]

    candidate = pred_full_placement if parsed_ok else baseline_placement
    valid, _checks = check_candidate_placement(candidate, services, NODES, LINKS, gt_reqs, flows)

    rows.append({
        "baseline": rc["baseline"],
        "model": rc["model"], "model_short": MODEL_SHORT_NAME[rc["model"]],
        "strategy": rc["prompt_strategy"], "scenario_id": r["scenario_id"],
        "parsed_ok": parsed_ok,
        "pred_decision": pred_decision, "gt_decision": gt_decision,
        "decision_correct": (pred_decision == gt_decision) if parsed_ok else False,
        "n_moved_pred": len(pred_moved), "n_moved_gt": len(gt_moved),
        "moved_correct": (pred_moved == gt_moved) if (parsed_ok and gt_decision == "OFFLOAD") else None,
        "real_valid": valid if parsed_ok else False,
    })

df = pd.DataFrame(rows)
df.shape


## Succes de parsing, par baseline x modele x strategie

In [ ]:
parse_rate = df.groupby(["baseline", "model_short", "strategy"])["parsed_ok"].mean().unstack("strategy")
display(parse_rate.reindex(index=pd.MultiIndex.from_product([["direct", "cot"], MODEL_ORDER]))[STRATEGY_ORDER].round(3))


## Exactitude de la decision STAY/OFFLOAD/INFEASIBLE

In [ ]:
decision_acc = df.groupby(["baseline", "model_short", "strategy"])["decision_correct"].mean().unstack("strategy")
display(decision_acc.reindex(index=pd.MultiIndex.from_product([["direct", "cot"], MODEL_ORDER]))[STRATEGY_ORDER].round(3))


## Validite REELLE du placement (contre les vraies exigences)

INFEASIBLE est exclu -- par construction, aucune configuration ne peut etre
valide quand la verite terrain dit qu'aucune ne l'est (voir le notebook
principal pour la verification independante de cette regle).


In [ ]:
real_valid_by_decision = df.groupby(["baseline", "model_short", "strategy", "gt_decision"])["real_valid"].mean().unstack("gt_decision")
display(real_valid_by_decision.reindex(index=pd.MultiIndex.from_product([["direct", "cot"], MODEL_ORDER, STRATEGY_ORDER]))[["STAY", "OFFLOAD", "INFEASIBLE"]].round(3))


## Exactitude de l'ensemble des services deplaces (seulement si verite terrain = OFFLOAD)

In [ ]:
moved_acc = df[df["gt_decision"] == "OFFLOAD"].groupby(["baseline", "model_short", "strategy"])["moved_correct"].mean().unstack("strategy")
display(moved_acc.reindex(index=pd.MultiIndex.from_product([["direct", "cot"], MODEL_ORDER]))[STRATEGY_ORDER].round(3))


## Matrices de confusion de la decision, par baseline x modele (strategies confondues)

In [ ]:
for baseline in ["direct", "cot"]:
    for model in MODEL_ORDER:
        sub = df[(df["baseline"] == baseline) & (df["model_short"] == model)]
        cm = pd.crosstab(sub["gt_decision"], sub["pred_decision"], rownames=["ground truth"], colnames=["predit"], dropna=False)
        print(f"--- {baseline} / {model} ---")
        display(cm)


## Vue d'ensemble -- tableau recapitulatif

In [ ]:
summary = pd.DataFrame({
    "parse_rate": df.groupby(["baseline", "model_short", "strategy"])["parsed_ok"].mean(),
    "decision_accuracy": df.groupby(["baseline", "model_short", "strategy"])["decision_correct"].mean(),
    "moved_accuracy_when_offload": df[df["gt_decision"] == "OFFLOAD"].groupby(["baseline", "model_short", "strategy"])["moved_correct"].mean(),
    "real_valid_stay": df[df["gt_decision"] == "STAY"].groupby(["baseline", "model_short", "strategy"])["real_valid"].mean(),
    "real_valid_offload": df[df["gt_decision"] == "OFFLOAD"].groupby(["baseline", "model_short", "strategy"])["real_valid"].mean(),
})
display(summary.round(3))


## Figure -- exactitude de la decision, par baseline x modele x strategie

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = ROOT / "eval" / "baselines" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def plot_metric_by_baseline(col, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    for ax, baseline in zip(axes, ["direct", "cot"]):
        pivot = df[df["baseline"] == baseline].groupby(["model_short", "strategy"])[col].mean().unstack("strategy")
        pivot = pivot.reindex(index=MODEL_ORDER, columns=STRATEGY_ORDER)
        pivot.plot.bar(ax=ax, color=[STRATEGY_COLORS[s] for s in STRATEGY_ORDER], width=0.75, edgecolor="white", linewidth=0.6)
        ax.set_title(f"{title} -- {baseline}")
        ax.set_ylabel(title)
        ax.set_xlabel("")
        ax.set_ylim(0, 1)
        ax.legend([STRATEGY_LABELS[s] for s in STRATEGY_ORDER], loc="lower right", frameon=True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Sauvegarde : {FIGURES_DIR / filename}")


plot_metric_by_baseline("decision_correct", "Exactitude de la decision", "decision_accuracy.png")


In [ ]:
plot_metric_by_baseline("parsed_ok", "Taux de parsing reussi", "parse_rate.png")
